In [ ]:
import pandas as pd
import geopandas as gpd
import networkx as nx
import folium
from shapely.geometry import Point
from pathlib import Path

# load roads with risk
segment_risk = pd.read_csv("../data/processed/segment_risk.csv")
roads = gpd.read_file("../data/external/nyc_roads.gpkg")

roads['physicalid'] = roads['physicalid'].astype(str)
segment_risk['physicalid'] = segment_risk['physicalid'].astype(str)

roads_risk = roads.merge(segment_risk[['physicalid','risk_class','total_csi']], 
                          on='physicalid', how='left')
roads_risk = roads_risk.to_crs("EPSG:3857")

# assign risk weight — safe route avoids high risk segments
risk_weight = {'Low': 1, 'Moderate': 3, 'High': 10}
roads_risk['weight'] = roads_risk['risk_class'].map(risk_weight).fillna(3)

print(f"Roads with risk weights: {len(roads_risk):,}")
print(roads_risk['risk_class'].value_counts())

Roads with risk weights: 122,272
risk_class
Low         44845
Moderate    12436
High         6384
Name: count, dtype: int64


In [4]:
from shapely.geometry import MultiLineString, LineString
print("Building road network graph...")

G = nx.Graph()

for _, row in roads_risk.iterrows():
    if row.geometry is None:
        continue
    
    # handle both LineString and MultiLineString
    if isinstance(row.geometry, MultiLineString):
        geoms = list(row.geometry.geoms)
    else:
        geoms = [row.geometry]
    
    for geom in geoms:
        coords = list(geom.coords)
        if len(coords) < 2:
            continue
        start = coords[0]
        end = coords[-1]
        length = geom.length
        weight = row['weight'] * length
        
        G.add_edge(start, end,
                   weight=weight,
                   risk_class=row['risk_class'],
                   street=row.get('street_name', ''),
                   geometry=geom)

print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

Building road network graph...
Graph nodes: 93,296
Graph edges: 134,963


In [5]:
from shapely.ops import nearest_points
from shapely.geometry import MultiPoint

def find_nearest_node(G, lat, lon):
    # convert to EPSG:3857
    point = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326")
    point = point.to_crs("EPSG:3857")
    px, py = point.geometry[0].x, point.geometry[0].y
    
    # find nearest graph node
    nearest = min(G.nodes(), key=lambda n: (n[0]-px)**2 + (n[1]-py)**2)
    return nearest

def safe_route(G, start_lat, start_lon, end_lat, end_lon):
    start_node = find_nearest_node(G, start_lat, start_lon)
    end_node   = find_nearest_node(G, end_lat, end_lon)
    
    try:
        path = nx.shortest_path(G, start_node, end_node, weight='weight')
        return path
    except nx.NetworkXNoPath:
        return None

# Test route — Times Square to Brooklyn Bridge
print("Finding safe route: Times Square → Brooklyn Bridge")
path = safe_route(G, 40.7580, -73.9855, 40.7061, -73.9969)

if path:
    print(f"Route found with {len(path)} waypoints")
else:
    print("No path found")

Finding safe route: Times Square → Brooklyn Bridge
Route found with 128 waypoints


In [6]:
if path:
    m = folium.Map(location=[40.7300, -74.0000], zoom_start=13, 
                   tiles='CartoDB positron')
    
    # draw route
    route_coords = [(node[1], node[0]) for node in path]  # lat, lon for folium
    
    # convert back to lat/lon
    route_gdf = gpd.GeoDataFrame(
        geometry=[Point(x, y) for x, y in path], crs="EPSG:3857"
    ).to_crs("EPSG:4326")
    
    route_latlon = [(geom.y, geom.x) for geom in route_gdf.geometry]
    
    folium.PolyLine(
        route_latlon, color='#2980b9', weight=5, opacity=0.9,
        tooltip="Safe Route"
    ).add_to(m)
    
    # start and end markers
    folium.Marker([40.7580, -73.9855], 
                  popup="Start: Times Square",
                  icon=folium.Icon(color='green', icon='play')).add_to(m)
    folium.Marker([40.7061, -73.9969], 
                  popup="End: Brooklyn Bridge",
                  icon=folium.Icon(color='red', icon='stop')).add_to(m)
    
    m.save("../outputs/maps/safe_route.html")
    print("Map saved. Open outputs/maps/safe_route.html")

Map saved. Open outputs/maps/safe_route.html


In [1]:
import pandas as pd
import geopandas as gpd
import json
import numpy as np

print("Preparing road network for export...")

segment_risk = pd.read_csv("../data/processed/segment_risk.csv")
roads = gpd.read_file("../data/external/nyc_roads.gpkg")

roads['physicalid'] = roads['physicalid'].astype(str)
segment_risk['physicalid'] = segment_risk['physicalid'].astype(str)

roads_risk = roads.merge(
    segment_risk[['physicalid','risk_class','total_csi']], 
    on='physicalid', how='left'
)
roads_risk = roads_risk.to_crs("EPSG:4326")
roads_risk['risk_class'] = roads_risk['risk_class'].fillna('Low')

# keep only high and moderate for overlay — too many low risk roads to render
display_roads = roads_risk[roads_risk['risk_class'].isin(['High','Moderate'])].copy()

# build GeoJSON for map overlay
features = []
for _, row in display_roads.iterrows():
    if row.geometry is None:
        continue
    try:
        features.append({
            "type": "Feature",
            "geometry": row.geometry.__geo_interface__,
            "properties": {
                "risk": row['risk_class'],
                "street": str(row.get('street_name', '')),
                "csi": float(row['total_csi']) if pd.notna(row['total_csi']) else 0
            }
        })
    except:
        continue

geojson = {"type": "FeatureCollection", "features": features}

with open("../data/external/roads_risk.geojson", "w") as f:
    json.dump(geojson, f)

print(f"Exported {len(features):,} road segments (High + Moderate only)")

# export full network as graph edges for routing
edges = []
for _, row in roads_risk.iterrows():
    if row.geometry is None:
        continue
    try:
        from shapely.geometry import MultiLineString
        geoms = list(row.geometry.geoms) if isinstance(
            row.geometry, MultiLineString) else [row.geometry]
        for geom in geoms:
            coords = list(geom.coords)
            if len(coords) < 2:
                continue
            edges.append({
                "start": [coords[0][1], coords[0][0]],   # lat, lon
                "end":   [coords[-1][1], coords[-1][0]],
                "coords": [[c[1], c[0]] for c in coords],
                "risk": row['risk_class'],
                "length": float(geom.length),
                "street": str(row.get('street_name', ''))
            })
    except:
        continue

with open("../data/external/graph_edges.json", "w") as f:
    json.dump(edges, f)

print(f"Exported {len(edges):,} graph edges for routing")

Preparing road network for export...
Exported 18,820 road segments (High + Moderate only)
Exported 136,704 graph edges for routing


In [4]:
html = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>NYC Safe Route Finder</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: 'Segoe UI', sans-serif; background: #f5f6fa; color: #2d3436; }

#header {
    background: white;
    padding: 14px 24px;
    border-bottom: 1px solid #dfe6e9;
    display: flex;
    align-items: center;
    justify-content: space-between;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
}
#header h1 { font-size: 16px; font-weight: 600; color: #2d3436; }
#header p  { font-size: 12px; color: #636e72; margin-top: 2px; }
.tag {
    background: #f0f4ff;
    color: #4a6cf7;
    padding: 4px 10px;
    border-radius: 20px;
    font-size: 11px;
    font-weight: 600;
}

#main { display: flex; height: calc(100vh - 57px); }
#map  { flex: 1; }

#panel {
    width: 300px;
    background: white;
    border-left: 1px solid #dfe6e9;
    display: flex;
    flex-direction: column;
    overflow-y: auto;
}

.panel-section {
    padding: 16px;
    border-bottom: 1px solid #f0f0f0;
}

.section-label {
    font-size: 10px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 1px;
    color: #b2bec3;
    margin-bottom: 10px;
}

.instruction {
    font-size: 13px;
    color: #636e72;
    line-height: 1.8;
}
.instruction b { color: #2d3436; }

.btn {
    width: 100%;
    padding: 10px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
    font-size: 13px;
    font-weight: 600;
    margin-bottom: 8px;
    transition: all 0.2s;
}
#btn-find {
    background: #2d3436;
    color: white;
}
#btn-find:hover { background: #4a6cf7; }
#btn-clear {
    background: #f5f6fa;
    color: #636e72;
    border: 1px solid #dfe6e9;
}
#btn-clear:hover { background: #dfe6e9; }

#status {
    font-size: 12px;
    color: #636e72;
    padding: 8px 10px;
    background: #f5f6fa;
    border-radius: 6px;
    border-left: 3px solid #4a6cf7;
    margin-top: 4px;
}

.route-card {
    background: #f5f6fa;
    border-radius: 8px;
    padding: 12px 14px;
    margin-bottom: 10px;
    border-left: 3px solid #dfe6e9;
}
.route-card.safe     { border-left-color: #00b894; }
.route-card.short    { border-left-color: #4a6cf7; }
.route-card.balanced { border-left-color: #fdcb6e; }

.route-title {
    font-size: 12px;
    font-weight: 700;
    margin-bottom: 8px;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
.route-card.safe .route-title     { color: #00b894; }
.route-card.short .route-title    { color: #4a6cf7; }
.route-card.balanced .route-title { color: #e17055; }

.best-tag {
    background: #00b894;
    color: white;
    font-size: 9px;
    padding: 2px 7px;
    border-radius: 10px;
    font-weight: 700;
}

.route-stat {
    display: flex;
    justify-content: space-between;
    font-size: 12px;
    color: #636e72;
    padding: 2px 0;
}
.route-stat span:last-child { font-weight: 600; color: #2d3436; }

.legend-item {
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 12px;
    color: #636e72;
    margin-bottom: 6px;
}
.legend-line {
    width: 24px;
    height: 3px;
    border-radius: 2px;
}
</style>
</head>
<body>

<div id="header">
    <div>
        <h1>NYC Safe Route Finder</h1>
        <p>Urban Crash Analytics — Road Collision Risk Prediction</p>
    </div>
    <span class="tag">ML-Powered</span>
</div>

<div id="main">
<div id="map"></div>
<div id="panel">

    <div class="panel-section">
        <div class="section-label">Instructions</div>
        <div class="instruction">
            <b>1.</b> Click map to set start<br>
            <b>2.</b> Click again to set destination<br>
            <b>3.</b> Press Find Routes
        </div>
        <div id="status" style="margin-top:10px">Click on the map to set start point.</div>
    </div>

    <div class="panel-section">
        <button class="btn" id="btn-find" onclick="findRoutes()">Find Routes</button>
        <button class="btn" id="btn-clear" onclick="clearAll()">Clear</button>
    </div>

    <div class="panel-section" id="results-section" style="display:none">
        <div class="section-label">Route Comparison</div>
        <div id="results"></div>
    </div>

    <div class="panel-section">
        <div class="section-label">Map Legend</div>
        <div class="legend-item">
            <div class="legend-line" style="background:#e17055;height:3px"></div>
            High Risk Road
        </div>
        <div class="legend-item">
            <div class="legend-line" style="background:#fdcb6e;height:2px"></div>
            Moderate Risk Road
        </div>
        <div class="legend-item">
            <div class="legend-line" style="background:#00b894;height:4px"></div>
            Safest Route
        </div>
        <div class="legend-item">
            <div class="legend-line" style="background:#4a6cf7;height:4px"></div>
            Shortest Route
        </div>
        <div class="legend-item">
            <div class="legend-line" style="background:#fdcb6e;height:4px"></div>
            Balanced Route
        </div>
    </div>

</div>
</div>

<script>
const map = L.map('map').setView([40.7128, -74.0060], 12);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
    attribution: '© CartoDB', maxZoom: 19
}).addTo(map);

let startMarker = null, endMarker = null;
let routeLayers = [], clickCount = 0;

const graphEdges = GRAPH_EDGES_PLACEHOLDER;
const riskGeoJSON = RISK_GEOJSON_PLACEHOLDER;

// risk overlay
L.geoJSON(riskGeoJSON, {
    style: f => ({
        color: f.properties.risk === 'High' ? '#e17055' : '#fdcb6e',
        weight: f.properties.risk === 'High' ? 2.5 : 1.5,
        opacity: 0.6
    }),
    onEachFeature: (f, layer) => {
        if (f.properties.street)
            layer.bindTooltip(`${f.properties.street} — ${f.properties.risk} Risk`);
    }
}).addTo(map);

// click handler
map.on('click', e => {
    const {lat, lng} = e.latlng;
    if (clickCount === 0) {
        if (startMarker) map.removeLayer(startMarker);
        startMarker = L.circleMarker([lat, lng], {
            radius: 8, fillColor: '#00b894', color: 'white',
            weight: 2, fillOpacity: 1
        }).addTo(map).bindPopup('<b>Start</b>').openPopup();
        setStatus('Start set. Now click your destination.');
        clickCount = 1;
    } else if (clickCount === 1) {
        if (endMarker) map.removeLayer(endMarker);
        endMarker = L.circleMarker([lat, lng], {
            radius: 8, fillColor: '#e17055', color: 'white',
            weight: 2, fillOpacity: 1
        }).addTo(map).bindPopup('<b>Destination</b>').openPopup();
        setStatus('Destination set. Press Find Routes.');
        clickCount = 2;
    }
});

// build graph from edges
function buildGraph() {
    const nodes = {}, adj = {};
    graphEdges.forEach(e => {
        const sk = e.start[0].toFixed(5)+','+e.start[1].toFixed(5);
        const ek = e.end[0].toFixed(5)+','+e.end[1].toFixed(5);
        if (!nodes[sk]) nodes[sk] = e.start;
        if (!nodes[ek]) nodes[ek] = e.end;
        if (!adj[sk]) adj[sk] = [];
        if (!adj[ek]) adj[ek] = [];
        const rw = {'Low':1,'Moderate':3,'High':10}[e.risk]||1;
        const d = e.length;
        adj[sk].push({node:ek, w:rw*d, d, risk:e.risk, coords:e.coords, street:e.street});
        adj[ek].push({node:sk, w:rw*d, d, risk:e.risk, coords:[...e.coords].reverse(), street:e.street});
    });
    return {nodes, adj};
}

function nearestNode(nodes, lat, lon) {
    let best = null, bestD = Infinity;
    for (const k in nodes) {
        const n = nodes[k];
        const d = (n[0]-lat)**2 + (n[1]-lon)**2;
        if (d < bestD) { bestD = d; best = k; }
    }
    return best;
}

function dijkstra(adj, start, end, wFn) {
    const dist = {[start]:0}, prev = {}, prevE = {};
    const visited = new Set();
    const pq = [[0, start]];
    while (pq.length) {
        pq.sort((a,b)=>a[0]-b[0]);
        const [d, u] = pq.shift();
        if (visited.has(u)) continue;
        visited.add(u);
        if (u === end) break;
        for (const e of (adj[u]||[])) {
            const nd = d + wFn(e);
            if (nd < (dist[e.node]??Infinity)) {
                dist[e.node] = nd;
                prev[e.node] = u;
                prevE[e.node] = e;
                pq.push([nd, e.node]);
            }
        }
    }
    const path = []; let cur = end;
    while (cur && prev[cur]) { path.unshift(prevE[cur]); cur = prev[cur]; }
    return path;
}

function findRoutes() {
    if (!startMarker || !endMarker) {
        setStatus('Please set both start and destination first.'); return;
    }
    setStatus('Calculating routes...');
    clearRoutes();
    setTimeout(() => {
        const {nodes, adj} = buildGraph();
        const sl = startMarker.getLatLng();
        const el = endMarker.getLatLng();
        const sk = nearestNode(nodes, sl.lat, sl.lng);
        const ek = nearestNode(nodes, el.lat, el.lng);

        const safe     = dijkstra(adj, sk, ek, e => e.w);
        const shortest = dijkstra(adj, sk, ek, e => e.d);
        const balanced = dijkstra(adj, sk, ek,
            e => ({'Low':1,'Moderate':2,'High':5}[e.risk]||1) * e.d);

        if (!safe.length && !shortest.length) {
            setStatus('No route found. Try points closer together or within NYC.');
            return;
        }

        drawRoute(safe,     '#00b894', 5, 'Safest Route');
        drawRoute(shortest, '#4a6cf7', 4, 'Shortest Route');
        drawRoute(balanced, '#e17055', 4, 'Balanced Route');
        showStats(safe, shortest, balanced);
        setStatus('Done. See route comparison below.');
        document.getElementById('results-section').style.display = 'block';
    }, 50);
}

function routeStats(path) {
    let dist=0, riskScore=0, high=0;
    path.forEach(e => {
        dist += e.d;
        riskScore += ({'Low':1,'Moderate':3,'High':10}[e.risk]||1) * e.d * 111;
        if (e.risk==='High') high++;
    });
    return {
        dist: dist > 0 ? (dist * 111).toFixed(2) : '0.00',
        risk: Math.round(riskScore),
        high
    };
}

function drawRoute(path, color, weight, label) {
    if (!path.length) return;
    const coords = [];
    path.forEach(e => e.coords.forEach(c => coords.push(c)));
    const layer = L.polyline(coords, {color, weight, opacity:0.9})
        .addTo(map).bindTooltip(label);
    routeLayers.push(layer);
    if (coords.length) map.fitBounds(L.polyline(coords).getBounds(), {padding:[40,40]});
}

function showStats(safe, short, bal) {
    const ss = routeStats(safe);
    const sh = routeStats(short);
    const bs = routeStats(bal);
    document.getElementById('results').innerHTML = `
        <div class="route-card safe">
            <div class="route-title">Safest Route <span class="best-tag">RECOMMENDED</span></div>
            <div class="route-stat"><span>Distance</span><span>${ss.dist} km</span></div>
            <div class="route-stat"><span>Risk Score</span><span>${ss.risk}</span></div>
            <div class="route-stat"><span>High Risk Segments</span><span>${ss.high}</span></div>
        </div>
        <div class="route-card short">
            <div class="route-title">Shortest Route</div>
            <div class="route-stat"><span>Distance</span><span>${sh.dist} km</span></div>
            <div class="route-stat"><span>Risk Score</span><span>${sh.risk}</span></div>
            <div class="route-stat"><span>High Risk Segments</span><span>${sh.high}</span></div>
        </div>
        <div class="route-card balanced">
            <div class="route-title">Balanced Route</div>
            <div class="route-stat"><span>Distance</span><span>${bs.dist} km</span></div>
            <div class="route-stat"><span>Risk Score</span><span>${bs.risk}</span></div>
            <div class="route-stat"><span>High Risk Segments</span><span>${bs.high}</span></div>
        </div>`;
}

function clearRoutes() {
    routeLayers.forEach(l => map.removeLayer(l));
    routeLayers = [];
    document.getElementById('results').innerHTML = '';
    document.getElementById('results-section').style.display = 'none';
}

function clearAll() {
    clearRoutes();
    if (startMarker) map.removeLayer(startMarker);
    if (endMarker)   map.removeLayer(endMarker);
    startMarker = endMarker = null;
    clickCount = 0;
    setStatus('Click on the map to set start point.');
}

function setStatus(msg) { document.getElementById('status').textContent = msg; }
</script>
</body>
</html>"""

import json, random

with open("../data/external/graph_edges.json") as f:
    edges_data = json.load(f)

with open("../data/external/roads_risk.geojson") as f:
    geojson_data = json.load(f)

# use ALL edges for proper routing
html = html.replace("GRAPH_EDGES_PLACEHOLDER", json.dumps(edges_data))
html = html.replace("RISK_GEOJSON_PLACEHOLDER", json.dumps(geojson_data))

with open("../outputs/maps/route_finder.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved. Edges: {len(edges_data):,} | Risk segments: {len(geojson_data['features']):,}")

print("Open outputs/maps/route_finder.html in your browser.")

Saved. Edges: 136,704 | Risk segments: 18,820
Open outputs/maps/route_finder.html in your browser.
